In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ==========================================
# 1. IoU (Intersection over Union) 계산 함수
# ==========================================
def compute_iou(box1, box2):
    """
    두 바운딩 박스 간의 IoU(교집합 넓이 / 합집합 넓이)를 계산하는 함수입니다.
    
    Parameters:
        box1, box2: [x1, y1, x2, y2] 형식의 바운딩 박스 좌표
                    (x1, y1): 좌상단 좌표, (x2, y2): 우하단 좌표
    Returns:
        float: 0.0 ~ 1.0 사이의 IoU 값
    """
    # 1-1. 두 박스가 중첩(교집합)되는 영역의 좌상단(ix1, iy1) 및 우하단(ix2, iy2) 좌표 구하기
    ix1 = max(box1[0], box2[0])  # 교집합 좌상단 x : 두 박스의 x1 중 더 큰 값
    iy1 = max(box1[1], box2[1])  # 교집합 좌상단 y : 두 박스의 y1 중 더 큰 값
    ix2 = min(box1[2], box2[2])  # 교집합 우하단 x : 두 박스의 x2 중 더 작은 값
    iy2 = min(box1[3], box2[3])  # 교집합 우하단 y : 두 박스의 y2 중 더 작은 값

    # 1-2. 교집합 영역의 너비(inter_w)와 높이(inter_h) 계산
    # 박스가 서로 겹치지 않으면 (ix2 - ix1)이 음수가 되므로 max(0, ...)를 사용하여 0으로 처리
    inter_w = max(0, ix2 - ix1)
    inter_h = max(0, iy2 - iy1)
    intersection = inter_w * inter_h  # 교집합 넓이

    # 1-3. 두 박스 각각의 넓이 및 전체 합집합(Union) 넓이 계산
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])  # 첫 번째 박스 전체 넓이
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])  # 두 번째 박스 전체 넓이
    
    # 합집합 넓이 = (박스1 넓이) + (박스2 넓이) - (중복된 교집합 넓이)
    union = area1 + area2 - intersection

    # 1-4. IoU 계산 (교집합 넓이 / 합집합 넓이)
    # 0으로 나누는 계산 오류(ZeroDivisionError)를 방지하기 위해 분모에 1e-8(아주 작은 값)을 더함
    return intersection / (union + 1e-8)


# ==========================================
# 2. NMS (Non-Maximum Suppression) 함수
# ==========================================
def nms(boxes, scores, iou_threshold=0.45):
    """
    중복으로 생성된 바운딩 박스를 제거하고 가장 신뢰도가 높은 박스만 남기는 함수입니다.
    
    Parameters:
        boxes: 바운딩 박스 목록 [[x1, y1, x2, y2], ...]
        scores: 각 박스에 대한 예측 신뢰도 점수 목록 [score1, score2, ...]
        iou_threshold: 중복으로 간주하여 제거할 IoU 임계값 (기본값: 0.45)
    Returns:
        list: 최종 선택(Keep)된 박스들의 인덱스 목록
    """
    # 2-1. 검사할 박스가 없는 경우 빈 리스트 반환
    if len(boxes) == 0:
        return []

    # 2-2. 신뢰도 점수(scores)를 기준 내림차순(높은 점수 -> 낮은 점수)으로 정렬했을 때의 원본 인덱스 추출
    # np.argsort는 오름차순이므로 [::-1]을 붙여 내림차순으로 뒤집음
    order = np.argsort(scores)[::-1]
    
    # 최종적으로 남겨둘 박스들의 인덱스를 담을 리스트
    keep = []

    # 2-3. 처리할 박스가 인덱스 배열(order)에 남아있는 동안 반복 수행
    while len(order) > 0:
        # 남아있는 박스 중 신뢰도 점수가 가장 높은 박스의 인덱스 선택
        best = order[0]
        keep.append(best)  # 해당 박스는 최종 채택 리스트에 추가

        # 남은 박스가 1개뿐이었다면 추가 비교 없이 반복 종료
        if len(order) == 1:
            break

        # 2-4. 현재 최고 점수 박스(best)와 나머지 박스들(order[1:]) 간의 IoU 일괄 계산
        ious = np.array(
            [compute_iou(boxes[best], boxes[i]) for i in order[1:]]
        )

        # 2-5. IoU가 임계값(iou_threshold) 미만인 박스만 남김
        # 즉, IoU >= iou_threshold인 '중복률이 높은 박스'들은 필터링되어 자동으로 제거됨
        order = order[1:][ious < iou_threshold]

    # 최종 선택된 박스 인덱스 목록 반환
    return keep


# ==========================================
# 3. 테스트 데이터 설정 및 NMS 실행
# ==========================================
# 테스트용 바운딩 박스 좌표 4개 [x1, y1, x2, y2]
test_boxes = [
    [100, 100, 300, 300],  # 박스 A (인덱스 0)
    [110, 110, 310, 310],  # 박스 B (인덱스 1) - 박스 A와 높은 중복도
    [400, 200, 600, 400],  # 박스 C (인덱스 2) - 독립적인 위치
    [105, 105, 305, 305],  # 박스 D (인덱스 3) - 박스 A와 높은 중복도
]

# 각 박스별 신뢰도 점수 (A, B, C, D 순서)
test_scores = [0.95, 0.87, 0.82, 0.79]

# 박스 시각화 시 구별을 위해 지정한 4가지 색상 및 라벨
box_colors = ['red', 'orange', 'green', 'yellow']
box_labels = ['Box A (0.95)', 'Box B (0.87)', 'Box C (0.82)', 'Box D (0.79)']

# IoU 임계값 0.5로 설정하여 NMS 실행
kept_idx = nms(test_boxes, test_scores, iou_threshold=0.5)


# ==========================================
# 4. Matplotlib을 활용한 시각화 (NMS 전/후 비교)
# ==========================================
# 1행 2열 구조의 서브플롯(그래프 화면) 생성
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ------------------------------------------
# [왼쪽 화면] NMS 적용 전: 4개 전체 박스 표시
# ------------------------------------------
ax1.set_xlim(0, 700)
ax1.set_ylim(500, 0)  # 이미지 좌표계 기준(위쪽이 y=0, 아래로 갈수록 y 증가)에 맞춰 y축 반전
ax1.set_title("1. Before NMS (All 4 Boxes)", fontsize=13, fontweight='bold')

# 4개의 입력 박스를 반복문으로 순회하며 그려줌
for i, (box, score) in enumerate(zip(test_boxes, test_scores)):
    x1, y1, x2, y2 = box
    width, height = x2 - x1, y2 - y1  # 박스의 너비와 높이 계산
    color = box_colors[i]             # 해당 박스에 지정된 색상 추출

    # 사각형 도형 객체 생성 (배경은 투명, 테두리는 지정 색상)
    rect = patches.Rectangle(
        (x1, y1), width, height,
        linewidth=2, edgecolor=color, facecolor='none'
    )
    ax1.add_patch(rect)  # 그래프에 사각형 추가
    
    # 박스 좌측 상단에 박스 이름과 점수 텍스트 표시
    ax1.text(
        x1 + 5, y1 + 20, box_labels[i],
        color=color, fontsize=9, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.6)  # 가독성을 위한 검은색 배경 상자
    )
ax1.grid(True, linestyle=':', alpha=0.5)  # 격자선 표시

# ------------------------------------------
# [오른쪽 화면] NMS 적용 후: 최종 채택된 박스만 표시
# ------------------------------------------
ax2.set_xlim(0, 700)
ax2.set_ylim(500, 0)  # y축 반전 (이미지 좌표계)
ax2.set_title(f"2. After NMS ({len(kept_idx)} Kept Boxes)", fontsize=13, fontweight='bold')

# NMS 결과로 최종 채택된 인덱스(kept_idx)에 해당하는 박스만 순회
for i in kept_idx:
    box = test_boxes[i]
    x1, y1, x2, y2 = box
    width, height = x2 - x1, y2 - y1
    color = box_colors[i]

    # 채택된 박스는 두꺼운 선(linewidth=3)으로 강조하여 그림
    rect = patches.Rectangle(
        (x1, y1), width, height,
        linewidth=3, edgecolor=color, facecolor='none'
    )
    ax2.add_patch(rect)
    
    # [KEEP] 태그와 함께 박스 라벨 표시
    ax2.text(
        x1 + 5, y1 + 20, f"[KEEP] {box_labels[i]}",
        color=color, fontsize=9, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.6)
    )
ax2.grid(True, linestyle=':', alpha=0.5)

# 레이아웃 간격 정돈 및 전체 화면 출력
plt.tight_layout()
plt.show()